In [ ]:
# Real-Time Object Detection & Classification System

Task : - Build a complete computer vision system that can detect and classify objects in images and video using YOLO v8, with additional features like image preprocessing and result analysis.



What You'll Apply -
1. Image Processing & OpenCV
2. Object Detection (YOLO)
3. Segmentation & Pose Estimation
4. CNN & Transfer Learning

In [ ]:
# Project: Multi-Feature CV Application

In [ ]:
!pip install ultralytics
!pip install opencv-python

In [ ]:
# ============================================
# 0. COLAB SETUP (RUN FIRST)
# ============================================
!pip install -q ultralytics opencv-python matplotlib tensorflow

import os
import urllib.request

# Create sample folder
os.makedirs("sample_images", exist_ok=True)

# Download open-source sample images (Ultralytics public images)
image_urls = [
    "https://ultralytics.com/images/bus.jpg",
    "https://ultralytics.com/images/zidane.jpg",
]

for url in image_urls:
    filename = os.path.join("sample_images", url.split("/")[-1])
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)

print("✅ Open-source images downloaded successfully!")

In [ ]:
# ============================================
# 1. IMPORTS & LOAD MODELS
# ============================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from collections import Counter
import time
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# Load YOLO models (weights auto-download)
det_model = YOLO('yolov8n.pt')
seg_model = YOLO('yolov8n-seg.pt')

print("✅ Models loaded successfully!")
print(f"Detection classes: {len(det_model.names)} classes")

In [ ]:
# ============================================
# 2. IMAGE PREPROCESSOR
# ============================================
class ImagePreprocessor:

    @staticmethod
    def resize_if_needed(img, max_size=1280):
        h, w = img.shape[:2]
        if max(h, w) > max_size:
            scale = max_size / max(h, w)
            img = cv2.resize(img, None, fx=scale, fy=scale)
        return img

preprocessor = ImagePreprocessor()

In [ ]:
# ============================================
# 3. OBJECT DETECTOR MODULE
# ============================================
class ObjectDetector:
    def __init__(self, model):
        self.model = model

    def detect(self, image_path, conf_threshold=0.5):
        img = cv2.imread(image_path)
        img = preprocessor.resize_if_needed(img)

        start_time = time.time()
        results = self.model(img, conf=conf_threshold)
        inference_time = time.time() - start_time

        return results, inference_time

    def analyze_results(self, results):
        analysis = {
            'total_objects': 0,
            'class_counts': Counter(),
            'detections': []
        }

        for result in results:
            for box in result.boxes:
                cls_id = int(box.cls[0].item())
                cls_name = self.model.names[cls_id]
                conf = box.conf[0].item()
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                analysis['total_objects'] += 1
                analysis['class_counts'][cls_name] += 1
                analysis['detections'].append({
                    'class': cls_name,
                    'confidence': conf,
                    'bbox': (x1, y1, x2, y2)
                })

        return analysis

    def visualize_results(self, image_path, results, analysis, inference_time):
        img = cv2.imread(image_path)
        img = preprocessor.resize_if_needed(img)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 3, figsize=(20, 6))

        axes[0].imshow(img_rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        annotated = results[0].plot()
        axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Detections ({analysis["total_objects"]} objects, {inference_time:.3f}s)')
        axes[1].axis('off')

        if analysis['class_counts']:
            classes = list(analysis['class_counts'].keys())
            counts = list(analysis['class_counts'].values())
            axes[2].barh(classes, counts)
            axes[2].set_title('Class Distribution')
        else:
            axes[2].text(0.5, 0.5, 'No objects detected', ha='center')

        plt.tight_layout()
        plt.show()

    def detect_and_report(self, image_path, conf_threshold=0.5):
        results, inference_time = self.detect(image_path, conf_threshold)
        analysis = self.analyze_results(results)

        print("\n==============================")
        print("📊 DETECTION REPORT")
        print("==============================")
        print(f"Inference Time: {inference_time:.3f}s")
        print(f"Total Objects: {analysis['total_objects']}")

        for cls, count in analysis['class_counts'].most_common():
            print(f"{cls}: {count}")

        self.visualize_results(image_path, results, analysis, inference_time)

detector = ObjectDetector(det_model)

In [ ]:
# ============================================
# 4. RUN DETECTION ON OPEN SOURCE IMAGES
# ============================================
for img_file in os.listdir("sample_images"):
    path = os.path.join("sample_images", img_file)
    print(f"\nProcessing: {img_file}")
    detector.detect_and_report(path)

In [ ]:
# ============================================
# 5. SEGMENTATION DEMO (OPEN SOURCE IMAGE)
# ============================================
class SegmentationModule:
    def __init__(self, model):
        self.model = model

    def segment(self, image_path):
        results = self.model(image_path)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        img = cv2.imread(image_path)
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original')
        axes[0].axis('off')

        annotated = results[0].plot()
        axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title('Instance Segmentation')
        axes[1].axis('off')

        plt.show()

segmenter = SegmentationModule(seg_model)

# Run segmentation on one open-source image
segmenter.segment("sample_images/bus.jpg")

In [ ]:
# ============================================
# 6. TRANSFER LEARNING CLASSIFIER (OPEN SOURCE)
# ============================================
def build_custom_classifier(num_classes, input_shape=(224, 224, 3)):
    base_model = MobileNetV2(input_shape=input_shape,
                             include_top=False,
                             weights='imagenet')
    base_model.trainable = False

    model = tf.keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

custom_model = build_custom_classifier(num_classes=5)
custom_model.summary()